# 04 · Train on Google Colab (free GPU)

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. When the upload button appears, choose `stress-detection.zip` (this project zipped).

The notebook downloads WESAD (2.25 GB, ~2–5 min on Colab), builds the windows, trains the
Random-Forest baseline, the CNN-LSTM and the hybrid CNN-LSTM + features model with
**leave-one-subject-out** evaluation, and finally
downloads `results.zip` with all metrics, plots and model weights.

In [ ]:
# Which experiments to run. With ENSEMBLE = 5, each deep LOSO run takes ~10-20 min on a T4;
# set ENSEMBLE = 1 for a quick (noisier) pass.
TASKS = ['4class', 'binary']        # add '3class' if you have time
ENSEMBLE = 5
WESAD_URL = 'https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download'

## 1 · Get the code

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

PROJECT = Path('/content/stress-detection')
if not (PROJECT / 'src' / 'train.py').exists():
    from google.colab import files
    print('Upload stress-detection.zip ...')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall('/content/_upload')
    src_dir = Path(glob.glob('/content/_upload/**/src/train.py', recursive=True)[0]).parents[1]
    shutil.move(str(src_dir), PROJECT)
os.chdir(PROJECT)
print('Working in', os.getcwd()); print(os.listdir())

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU - check runtime type!)')

## 2 · Download WESAD

In [ ]:
if not glob.glob('WESAD/S*/S*.pkl'):
    !wget -q --show-progress -O /content/WESAD.zip "$WESAD_URL"
    # only the per-subject pickles are needed
    !unzip -q -o /content/WESAD.zip 'WESAD/S*/S*.pkl' -d .
    !rm -f /content/WESAD.zip
print(sorted(glob.glob('WESAD/S*/S*.pkl')))

## 3 · Preprocess

In [ ]:
!python -m src.data

## 4 · Train

1. **Random Forest** on hand-crafted features (HR, HRV, SCR count, breathing rate, …)
2. **CNN-LSTM** on the raw 60 s signals
3. **Hybrid**: CNN-LSTM signal embedding fused with the hand-crafted features

Deep models average `ENSEMBLE` networks trained with different seeds, because a single run varies by about ±0.05 macro-F1 from the seed alone.

In [ ]:
for task in TASKS:
    !python -m src.train --model rf --task {task} --eval loso

In [ ]:
for task in TASKS:
    !python -m src.train --model cnnlstm --task {task} --eval loso --ensemble {ENSEMBLE} --tag final

In [ ]:
for task in TASKS:
    !python -m src.train --model hybrid --task {task} --eval loso --ensemble {ENSEMBLE} --tag final

In [ ]:
# Fixed split (same test subjects as v1) - also saves model weights to models/
!python -m src.train --model cnnlstm --task 4class --eval split --verbose

## 5 · Results

In [ ]:
!python -m src.report
import pandas as pd
pd.read_csv('outputs/summary.csv').set_index('run')

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('outputs/figures/*.png')) + sorted(glob.glob('outputs/*_final/confusion_matrix.png')):
    print(f); display(Image(f, width=900))

In [ ]:
!zip -q -r /content/results.zip outputs models
from google.colab import files
files.download('/content/results.zip')